Хочу, чтобы этот ноутбук читался как рабочая тетрадь, а не как сухой туториал.

Идея простая: берем конфиг, быстро собираем надежный dense-препроцессинг, пробуем несколько моделей, потом руками собираем стекинг и сразу делаем инференс.

Если где-то всплывает старый traceback с `HistGradientBoostingRegressor` или старой сигнатурой `evaluate_model(name, estimator, preprocessor)`, значит kernel держит старую версию файла.

In [ ]:
from pathlib import Path
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from omegaconf import OmegaConf
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.model_selection import KFold, cross_val_score

from config import py_config

if isinstance(py_config, dict):
    cfg_dict = py_config.copy()
else:
    cfg_dict = OmegaConf.to_container(py_config, resolve=False)

print('Notebook version: brainstorm_dense_v1')
sns.set_theme(style='whitegrid')

Сначала просто проверяю, что конфиг живой и действительно управляет путями. Это важно: если данные читаются не из конфига, потом легко потерять контроль над пайплайном.

In [ ]:
train_path = Path(cfg_dict['paths']['train'])
test_path = Path(cfg_dict['paths']['test'])
artifacts_dir = Path(cfg_dict['paths']['artifacts_dir'])
target_col = cfg_dict['data']['target_col']
id_col = cfg_dict['data']['id_col']

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train_raw = train_df[target_col].copy()
X_test_raw = test_df.copy()

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
train_df.head()

Обычно перед моделями хочу глазами понять две вещи: как ведет себя таргет и насколько много пропусков. Этого уже достаточно, чтобы выбрать спокойную стратегию препроцессинга.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(y_train_raw, kde=True, ax=axes[0])
axes[0].set_title('target')

missing_counts = train_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0].head(cfg_dict['eda']['top_numeric_features'])
sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[1])
axes[1].set_title('missing')

plt.tight_layout()
plt.show()

missing_counts.to_frame('missing_count')

Дальше мысль такая: не надо сразу лезть в сложные пайплайны. Сначала добавим несколько человеческих признаков из конфига и сделаем это максимально прозрачно.

In [ ]:
def apply_feature_engineering(df, cfg):
    df = df.copy()
    fe_cfg = cfg['feature_engineering']

    if not fe_cfg.get('enabled', True):
        return df

    if fe_cfg.get('add_time_features', True):
        if {'YrSold', 'MoSold'}.issubset(df.columns):
            df['SaleMonthIndex'] = df['YrSold'] * 12 + df['MoSold']
        if 'YrSold' in df.columns:
            df['YearsSinceSale2010'] = 2010 - df['YrSold']

    if fe_cfg.get('add_total_area', True):
        area_parts = [c for c in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'GarageArea'] if c in df.columns]
        if area_parts:
            df['TotalUsableArea'] = df[area_parts].fillna(0).sum(axis=1)

    if fe_cfg.get('add_total_bathrooms', True):
        bath_map = {
            'FullBath': 1.0,
            'HalfBath': 0.5,
            'BsmtFullBath': 1.0,
            'BsmtHalfBath': 0.5,
        }
        existing_baths = [c for c in bath_map if c in df.columns]
        if existing_baths:
            df['TotalBathrooms'] = sum(df[c].fillna(0) * bath_map[c] for c in existing_baths)

    if fe_cfg.get('add_house_age', True):
        if {'YrSold', 'YearBuilt'}.issubset(df.columns):
            df['HouseAgeAtSale'] = df['YrSold'] - df['YearBuilt']
        if {'YrSold', 'YearRemodAdd'}.issubset(df.columns):
            df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

    if fe_cfg.get('add_interactions', True):
        for left, right in fe_cfg.get('interaction_pairs', []):
            if left in df.columns and right in df.columns:
                df[f'{left}_x_{right}'] = df[left].fillna(0) * df[right].fillna(0)

    return df


feature_preview_df = apply_feature_engineering(X_train_raw.head(5), cfg_dict)
feature_preview_df.head()

Теперь ключевое инженерное решение: делаем все в dense через `pandas`. Это не самый изящный способ в проде, но в учебной среде он надежнее и лучше объясняет, что происходит.

In [ ]:
def prepare_dense_datasets(X_train_raw, X_test_raw, cfg):
    combined_df = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
    combined_df = apply_feature_engineering(combined_df, cfg)
    engineered_df = combined_df.copy()

    numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in combined_df.columns if c not in numeric_cols]

    for col in numeric_cols:
        combined_df[col] = combined_df[col].fillna(combined_df[col].median())

    for col in categorical_cols:
        combined_df[col] = combined_df[col].fillna('Missing').astype(str)

    combined_df = pd.get_dummies(combined_df, drop_first=False)
    combined_df = combined_df.astype(float)

    X_ready_dense = combined_df.iloc[:len(X_train_raw)].copy()
    X_test_ready_dense = combined_df.iloc[len(X_train_raw):].copy()
    return X_ready_dense, X_test_ready_dense, engineered_df


X_ready_dense, X_test_ready_dense, engineered_features_df = prepare_dense_datasets(X_train_raw, X_test_raw, cfg_dict)

print('prepared train:', X_ready_dense.shape)
print('prepared test:', X_test_ready_dense.shape)
X_ready_dense.head()

Дальше нужна одна метрика и один сплиттер. Иначе сравнение моделей будет нечестным. Для цен домов удобно смотреть на RMSLE.

In [ ]:
def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))))


rmsle_scorer = make_scorer(rmsle, greater_is_better=False)
cv_splitter = KFold(
    n_splits=cfg_dict['validation']['primary_n_splits'],
    shuffle=cfg_dict['validation']['shuffle'],
    random_state=cfg_dict['seed'],
)

print('folds:', cfg_dict['validation']['primary_n_splits'])

Теперь фильтрую модели. Беру только те, которые реально нужны для разбора и точно стабильно отрабатывают в sklearn без внешних зависимостей.

In [ ]:
SUPPORTED_MODEL_BUILDERS = {
    'ridge': Ridge,
    'lasso': Lasso,
    'elasticnet': ElasticNet,
    'random_forest': RandomForestRegressor,
}


def filter_params_for_estimator(estimator_cls, params):
    valid_names = set(inspect.signature(estimator_cls.__init__).parameters)
    return {key: value for key, value in params.items() if key in valid_names}


def build_estimator(model_name, params):
    estimator_cls = SUPPORTED_MODEL_BUILDERS[model_name]
    clean_params = filter_params_for_estimator(estimator_cls, params)
    return estimator_cls(**clean_params)


active_supported_models = []
skipped_models = []

for model_name, model_cfg in cfg_dict['models'].items():
    if not model_cfg.get('enabled', False):
        skipped_models.append((model_name, 'disabled'))
        continue
    if model_name not in SUPPORTED_MODEL_BUILDERS:
        skipped_models.append((model_name, 'skip here, too much noise for learning'))
        continue

    first_grid = model_cfg.get('grids', [{}])[0] if model_cfg.get('grids') else {}
    active_supported_models.append({
        'name': model_name,
        'params': first_grid,
        'estimator': build_estimator(model_name, first_grid),
    })

print('active:', [item['name'] for item in active_supported_models])
print('skipped:', skipped_models)
pd.DataFrame(skipped_models, columns=['model', 'reason'])

Хорошая привычка: одна обертка для target transform и одна функция для оценки. Тогда можно думать про модели, а не копировать один и тот же код.

In [ ]:
def wrap_target_transform(estimator, cfg):
    if cfg['data']['target_transform'] == 'log1p':
        return TransformedTargetRegressor(
            regressor=estimator,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    return estimator


def evaluate_model(model_name, estimator, X, y, cfg):
    model = wrap_target_transform(clone(estimator), cfg)
    scores = -cross_val_score(
        model,
        X,
        y,
        scoring=rmsle_scorer,
        cv=cv_splitter,
        n_jobs=1,
        error_score='raise',
    )
    return {
        'model': model_name,
        'params': str(estimator.get_params()),
        'mean_rmsle': float(scores.mean()),
        'std_rmsle': float(scores.std()),
    }


Теперь просто прогоняю baseline. Это полезно: прежде чем строить ансамбль, надо понять, кто из одиночных моделей вообще уже тянет задачу.

In [ ]:
baseline_results = []
for item in active_supported_models:
    baseline_results.append(
        evaluate_model(item['name'], item['estimator'], X_ready_dense, y_train_raw, cfg_dict)
    )

baseline_results_df = pd.DataFrame(baseline_results).sort_values('mean_rmsle').reset_index(drop=True)
baseline_results_df

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=baseline_results_df, x='mean_rmsle', y='model')
plt.title('baseline')
plt.xlabel('mean cv rmsle')
plt.ylabel('model')
plt.show()

А теперь самое полезное для понимания: ручной стекинг.

Почему руками? Потому что так видно механику: базовые модели дают OOF-предсказания, эти предсказания становятся новыми признаками, а сверху учится meta-модель.

In [ ]:
def fit_manual_stacking(X, y, X_test, cfg, active_models):
    preferred_order = ['ridge', 'elasticnet', 'random_forest', 'lasso']
    available = {item['name']: item for item in active_models}
    selected_names = []

    for name in preferred_order:
        if name in available and len(selected_names) < 3:
            selected_names.append(name)

    if len(selected_names) < 2:
        raise ValueError('Need at least two models for stacking.')

    selected_models = [available[name] for name in selected_names]
    fold_indices = list(cv_splitter.split(X, y))
    stacking_oof_matrix = np.zeros((len(X), len(selected_models)), dtype=float)
    stacking_test_matrix = np.zeros((len(X_test), len(selected_models)), dtype=float)

    for model_idx, model_info in enumerate(selected_models):
        fold_test_predictions = []

        for train_idx, valid_idx in fold_indices:
            base_estimator = wrap_target_transform(clone(model_info['estimator']), cfg)
            base_estimator.fit(X.iloc[train_idx], y.iloc[train_idx])
            valid_pred = base_estimator.predict(X.iloc[valid_idx])
            stacking_oof_matrix[valid_idx, model_idx] = valid_pred
            fold_test_predictions.append(base_estimator.predict(X_test))

        stacking_test_matrix[:, model_idx] = np.mean(np.column_stack(fold_test_predictions), axis=1)

    meta_cfg = cfg['ensembles']['stacker_models'][0]
    meta_model = Ridge(**meta_cfg.get('params', {}))
    meta_model.fit(stacking_oof_matrix, y)

    stacking_oof_pred = meta_model.predict(stacking_oof_matrix)
    stacking_test_pred = meta_model.predict(stacking_test_matrix)

    return {
        'base_model_names': selected_names,
        'oof_meta': stacking_oof_matrix,
        'test_meta': stacking_test_matrix,
        'meta_model': meta_model,
        'stacking_oof_pred': stacking_oof_pred,
        'stacking_test_pred': stacking_test_pred,
        'stacking_rmsle': rmsle(y, stacking_oof_pred),
    }


stacking_artifacts = fit_manual_stacking(
    X_ready_dense,
    y_train_raw,
    X_test_ready_dense,
    cfg_dict,
    active_supported_models,
)

stacking_result_row = {
    'model': 'stacking_ridge_meta',
    'params': str({
        'base_models': stacking_artifacts['base_model_names'],
        'meta_model': cfg_dict['ensembles']['stacker_models'][0],
    }),
    'mean_rmsle': float(stacking_artifacts['stacking_rmsle']),
    'std_rmsle': 0.0,
}

stacking_result_row

После стекинга уже можно честно сравнить все вместе. Хорошая инженерная привычка: не предполагать, что ансамбль автоматически лучше.

In [ ]:
results_df = pd.concat([
    baseline_results_df,
    pd.DataFrame([stacking_result_row]),
], ignore_index=True)

results_df = results_df.sort_values('mean_rmsle').reset_index(drop=True)
results_df

Теперь инференс. Логика простая: если стекинг лучше, берем его. Если нет, берем лучшую одиночную модель. Это уже похоже на реальный рабочий decision point.

In [ ]:
best_baseline_row = baseline_results_df.iloc[0]
use_stacking = stacking_result_row['mean_rmsle'] <= float(best_baseline_row['mean_rmsle'])

if use_stacking:
    final_model_name = 'stacking_ridge_meta'
    final_predictions_array = np.clip(stacking_artifacts['stacking_test_pred'], 0, None)
else:
    final_model_name = best_baseline_row['model']
    baseline_model_info = next(item for item in active_supported_models if item['name'] == final_model_name)
    fitted_baseline_model = wrap_target_transform(clone(baseline_model_info['estimator']), cfg_dict)
    fitted_baseline_model.fit(X_ready_dense, y_train_raw)
    final_predictions_array = np.clip(fitted_baseline_model.predict(X_test_ready_dense), 0, None)

submission = pd.DataFrame({
    id_col: X_test_raw[id_col],
    target_col: final_predictions_array,
})

submission_path = Path(cfg_dict['output']['submission_path'])
submission_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(submission_path, index=False)

print('selected:', final_model_name)
print('saved to:', submission_path.resolve())
submission.head()

Ниже не unit-тесты, а быстрые sanity-check проверки. Они нужны, чтобы сразу поймать типовые поломки: потерянные строки, NaN после препроцессинга, сломанный submission.

In [ ]:
expected_engineered_columns = [
    'SaleMonthIndex',
    'YearsSinceSale2010',
    'TotalUsableArea',
    'TotalBathrooms',
    'HouseAgeAtSale',
]

assert train_path.exists()
assert test_path.exists()
assert target_col in train_df.columns
assert id_col in test_df.columns

assert len(X_ready_dense) == len(train_df)
assert len(X_test_ready_dense) == len(test_df)
assert X_ready_dense.shape[1] == X_test_ready_dense.shape[1]
assert not X_ready_dense.isna().any().any()
assert not X_test_ready_dense.isna().any().any()

if cfg_dict['feature_engineering']['enabled']:
    assert any(col in engineered_features_df.columns for col in expected_engineered_columns)

assert not results_df.empty
assert results_df['mean_rmsle'].notna().all()
assert np.isfinite(results_df['mean_rmsle']).all()

assert stacking_artifacts['oof_meta'].shape[0] == len(train_df)
assert stacking_artifacts['test_meta'].shape[0] == len(test_df)
assert np.isfinite(stacking_artifacts['stacking_oof_pred']).all()

assert len(final_predictions_array) == len(test_df)
assert np.isfinite(final_predictions_array).all()
assert (final_predictions_array >= 0).all()

assert submission.columns.tolist() == [id_col, target_col]
assert len(submission) == len(test_df)

print('All sanity checks passed.')

Финальная мысль.

Здесь специально выбран не самый “умный”, а самый понятный и устойчивый путь: сначала прозрачная подготовка, потом честное сравнение, потом ручной ансамбль, потом инференс и быстрая самопроверка. Именно так удобно разбирать мышление по шагам.